# RAGAS Inline Evaluation Test

- **LlamaStack**: `llama-stack-ragas-inline` in `ragas-eval` namespace
- **Inference model**: `Gemma-3-27B-BF16-Distributed` (vLLM in `vszp` namespace)
- **Embedding model**: `qwen3-4b-embedding` (vLLM in `vszp` namespace)
- **RAGAS provider**: `trustyai_ragas_inline`

## Setup and Imports

In [2]:
!pip install llama-stack-client==0.4.2 rich pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [llama-stack-client]lama-stack-client]


In [3]:
from datetime import datetime

import pandas as pd
from llama_stack_client import LlamaStackClient
from rich.pretty import pprint

## Connect to LlamaStack

In [4]:
LLAMA_STACK_URL = "http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321"
PROVIDER_ID_INLINE = "trustyai_ragas_inline"

client = LlamaStackClient(base_url=LLAMA_STACK_URL)

print("Available models:")
available_models = client.models.list()
pprint(available_models)

print("\nAvailable eval providers:")
providers = client.providers.list()
eval_providers = [p for p in providers if p.api == "eval"]
pprint(eval_providers)

Available models:


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


[
│   Model(
│   │   id='vllm-embedding/qwen3-4b-embedding',
│   │   created=1777977748,
│   │   owned_by='llama_stack',
│   │   custom_metadata={
│   │   │   'model_type': 'embedding',
│   │   │   'provider_id': 'vllm-embedding',
│   │   │   'provider_resource_id': 'qwen3-4b-embedding',
│   │   │   'embedding_dimension': 2048
│   │   },
│   │   object='model'
│   ),
│   Model(
│   │   id='vllm-inference/Gemma-3-27B-BF16-Distributed',
│   │   created=1777977748,
│   │   owned_by='llama_stack',
│   │   custom_metadata={
│   │   │   'model_type': 'llm',
│   │   │   'provider_id': 'vllm-inference',
│   │   │   'provider_resource_id': 'Gemma-3-27B-BF16-Distributed'
│   │   },
│   │   object='model'
│   )
]

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/providers "HTTP/1.1 200 OK"



Available eval providers:


[
│   ProviderInfo(
│   │   api='eval',
│   │   config={
│   │   │   'use_k8s': True,
│   │   │   'base_url': 'https://gemma-3-27b-bf16-distributed-vszp.apps.cluster-5pzpt.5pzpt.sandbox1134.opentlc.com/v1'
│   │   },
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_lmeval',
│   │   provider_type='remote::trustyai_lmeval'
│   ),
│   ProviderInfo(
│   │   api='eval',
│   │   config={'embedding_model': 'vllm-embedding/qwen3-4b-embedding'},
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_ragas_inline',
│   │   provider_type='inline::trustyai_ragas'
│   )
]

## Identify Models

Find the LLM and embedding model IDs from the registered models.

In [5]:
llm_model = None
embedding_model = None

for m in available_models:
    meta = m.custom_metadata or {}
    if meta.get("model_type") == "llm":
        llm_model = m.id
    elif meta.get("model_type") == "embedding":
        embedding_model = m.id

print(f"LLM model:       {llm_model}")
print(f"Embedding model: {embedding_model}")

assert llm_model is not None, "No LLM model found!"
assert embedding_model is not None, "No embedding model found!"

LLM model:       vllm-inference/Gemma-3-27B-BF16-Distributed
Embedding model: vllm-embedding/qwen3-4b-embedding


## Dataset Preparation

Sample RAG evaluation dataset (same as basic_demo).

In [6]:
evaluation_data = [
    {
        "user_input": "What is the capital of France?",
        "response": "The capital of France is Paris.",
        "retrieved_contexts": [
            "Paris is the capital and most populous city of France."
        ],
        "reference": "Paris",
    },
    {
        "user_input": "Who invented the telephone?",
        "response": "Alexander Graham Bell invented the telephone in 1876.",
        "retrieved_contexts": [
            "Alexander Graham Bell was a Scottish-American inventor who patented the first practical telephone."
        ],
        "reference": "Alexander Graham Bell",
    },
    {
        "user_input": "What is photosynthesis?",
        "response": "Photosynthesis is the process by which plants convert sunlight into energy.",
        "retrieved_contexts": [
            "Photosynthesis is a process used by plants to convert light energy into chemical energy."
        ],
        "reference": "Photosynthesis is the process by which plants and other organisms convert light energy into chemical energy.",
    },
]

## Register Dataset

In [7]:
dataset_id = "ragas_inline_test_dataset"

# Clean up if exists
try:
    client.beta.datasets.unregister(dataset_id)
except Exception:
    pass

dataset_response = client.beta.datasets.register(
    dataset_id=dataset_id,
    purpose="eval/question-answer",
    source={"type": "rows", "rows": evaluation_data},
    metadata={
        "provider_id": "localfs",
        "description": "Sample RAG evaluation dataset for inline RAGAS test",
        "size": len(evaluation_data),
        "format": "ragas",
        "created_at": datetime.now().isoformat(),
    },
)
pprint(dataset_response)

/tmp/ipykernel_95/3168177760.py:5: DeprecationWarning: deprecated
  client.beta.datasets.unregister(dataset_id)
INFO:httpx:HTTP Request: DELETE http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets/ragas_inline_test_dataset "HTTP/1.1 404 Not Found"
/tmp/ipykernel_95/3168177760.py:9: DeprecationWarning: deprecated
  dataset_response = client.beta.datasets.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets "HTTP/1.1 200 OK"


DatasetRegisterResponse(
│   identifier='ragas_inline_test_dataset',
│   provider_id='localfs',
│   purpose='eval/question-answer',
│   source=SourceRowsDataSource(
│   │   rows=[
│   │   │   {
│   │   │   │   'user_input': 'What is the capital of France?',
│   │   │   │   'response': 'The capital of France is Paris.',
│   │   │   │   'retrieved_contexts': ['Paris is the capital and most populous city of France.'],
│   │   │   │   'reference': 'Paris'
│   │   │   },
│   │   │   {
│   │   │   │   'user_input': 'Who invented the telephone?',
│   │   │   │   'response': 'Alexander Graham Bell invented the telephone in 1876.',
│   │   │   │   'retrieved_contexts': [
│   │   │   │   │   'Alexander Graham Bell was a Scottish-American inventor who patented the first practical telephone.'
│   │   │   │   ],
│   │   │   │   'reference': 'Alexander Graham Bell'
│   │   │   },
│   │   │   {
│   │   │   │   'user_input': 'What is photosynthesis?',
│   │   │   │   'response': 'Photosynthesis is the process by which plants convert sunlight into energy.',
│   │   │   │   'retrieved_contexts': [
│   │   │   │   │   'Photosynthesis is a process used by plants to convert light energy into chemical energy.'
│   │   │   │   ],
│   │   │   │   'reference': 'Photosynthesis is the process by which plants and other organisms convert light energy into chemical energy.'
│   │   │   }
│   │   ],
│   │   type='rows'
│   ),
│   metadata={
│   │   'provider_id': 'localfs',
│   │   'description': 'Sample RAG evaluation dataset for inline RAGAS test',
│   │   'size': 3,
│   │   'format': 'ragas',
│   │   'created_at': '2026-05-05T10:42:45.833920'
│   },
│   provider_resource_id='ragas_inline_test_dataset',
│   type='dataset'
)

## Register Benchmark (Inline Only)

In [9]:
benchmark_id = "ragas_inline_test_benchmark"

benchmark_response = client.alpha.benchmarks.register(
    benchmark_id=benchmark_id,
    dataset_id=dataset_id,
    scoring_functions=[
        "answer_relevancy",
    ],
    provider_id=PROVIDER_ID_INLINE,
)

print("Benchmark registered successfully.")
pprint(client.alpha.benchmarks.list())

/tmp/ipykernel_95/2641280209.py:3: DeprecationWarning: deprecated
  benchmark_response = client.alpha.benchmarks.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"


Benchmark registered successfully.


[
│   Benchmark(
│   │   dataset_id='smoke_test_dataset',
│   │   identifier='smoke_test_benchmark',
│   │   provider_id='trustyai_ragas_inline',
│   │   scoring_functions=['answer_relevancy'],
│   │   metadata={},
│   │   provider_resource_id='smoke_test_benchmark',
│   │   type='benchmark'
│   ),
│   Benchmark(
│   │   dataset_id='smoke_test_dataset',
│   │   identifier='smoke_test_benchmark_v2',
│   │   provider_id='trustyai_ragas_inline',
│   │   scoring_functions=['answer_relevancy'],
│   │   metadata={},
│   │   provider_resource_id='smoke_test_benchmark_v2',
│   │   type='benchmark'
│   ),
│   Benchmark(
│   │   dataset_id='ragas_inline_test_dataset',
│   │   identifier='ragas_inline_test_benchmark',
│   │   provider_id='trustyai_ragas_inline',
│   │   scoring_functions=['answer_relevancy'],
│   │   metadata={},
│   │   provider_resource_id='ragas_inline_test_benchmark',
│   │   type='benchmark'
│   )
]

## Run Inline Evaluation

In [10]:
inline_job = client.alpha.eval.run_eval(
    benchmark_id=benchmark_id,
    benchmark_config={
        "eval_candidate": {
            "type": "model",
            "model": llm_model,
            "sampling_params": {"temperature": 0.1, "max_tokens": 100},
        },
        "scoring_params": {},
    },
)
print("Job submitted:")
pprint(inline_job)

INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/ragas_inline_test_benchmark/jobs "HTTP/1.1 200 OK"


Job submitted:


Job(
│   job_id='1',
│   status='in_progress',
│   result=None,
│   eval_config={
│   │   'embedding_model': 'vllm-embedding/qwen3-4b-embedding',
│   │   'ragas_config': {
│   │   │   'batch_size': None,
│   │   │   'show_progress': True,
│   │   │   'raise_exceptions': True,
│   │   │   'experiment_name': None,
│   │   │   'column_map': None
│   │   }
│   }
)

## Check Job Status

Poll until the job completes.

In [11]:
import time

while True:
    status = client.alpha.eval.jobs.status(
        benchmark_id=benchmark_id, job_id=inline_job.job_id
    )
    print(f"Status: {status.status}")
    if status.status in ("completed", "failed"):
        break
    time.sleep(5)

pprint(status)

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/ragas_inline_test_benchmark/jobs/1 "HTTP/1.1 200 OK"


Status: completed


Job(
│   job_id='1',
│   status='completed',
│   result={
│   │   'generations': [
│   │   │   {
│   │   │   │   'user_input': 'What is the capital of France?',
│   │   │   │   'retrieved_contexts': ['Paris is the capital and most populous city of France.'],
│   │   │   │   'response': 'The capital of France is Paris.',
│   │   │   │   'reference': 'Paris'
│   │   │   },
│   │   │   {
│   │   │   │   'user_input': 'Who invented the telephone?',
│   │   │   │   'retrieved_contexts': [
│   │   │   │   │   'Alexander Graham Bell was a Scottish-American inventor who patented the first practical telephone.'
│   │   │   │   ],
│   │   │   │   'response': 'Alexander Graham Bell invented the telephone in 1876.',
│   │   │   │   'reference': 'Alexander Graham Bell'
│   │   │   },
│   │   │   {
│   │   │   │   'user_input': 'What is photosynthesis?',
│   │   │   │   'retrieved_contexts': [
│   │   │   │   │   'Photosynthesis is a process used by plants to convert light energy into chemical energy.'
│   │   │   │   ],
│   │   │   │   'response': 'Photosynthesis is the process by which plants convert sunlight into energy.',
│   │   │   │   'reference': 'Photosynthesis is the process by which plants and other organisms convert light energy into chemical energy.'
│   │   │   }
│   │   ],
│   │   'scores': {
│   │   │   'answer_relevancy': {
│   │   │   │   'score_rows': [{'score': 1.0}, {'score': 0.8741980651566914}, {'score': 0.999999999999999}],
│   │   │   │   'aggregated_results': {'answer_relevancy': 0.9580660217188969}
│   │   │   }
│   │   }
│   },
│   eval_config={
│   │   'embedding_model': 'vllm-embedding/qwen3-4b-embedding',
│   │   'ragas_config': {
│   │   │   'batch_size': None,
│   │   │   'show_progress': True,
│   │   │   'raise_exceptions': True,
│   │   │   'experiment_name': None,
│   │   │   'column_map': None
│   │   }
│   }
)

## Retrieve Results

In [12]:
inline_results = client.alpha.eval.jobs.retrieve(
    benchmark_id=benchmark_id, job_id=inline_job.job_id
)
pprint(inline_results)

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/ragas_inline_test_benchmark/jobs/1/result "HTTP/1.1 200 OK"


EvaluateResponse(
│   generations=[
│   │   {
│   │   │   'user_input': 'What is the capital of France?',
│   │   │   'retrieved_contexts': ['Paris is the capital and most populous city of France.'],
│   │   │   'response': 'The capital of France is Paris.',
│   │   │   'reference': 'Paris'
│   │   },
│   │   {
│   │   │   'user_input': 'Who invented the telephone?',
│   │   │   'retrieved_contexts': [
│   │   │   │   'Alexander Graham Bell was a Scottish-American inventor who patented the first practical telephone.'
│   │   │   ],
│   │   │   'response': 'Alexander Graham Bell invented the telephone in 1876.',
│   │   │   'reference': 'Alexander Graham Bell'
│   │   },
│   │   {
│   │   │   'user_input': 'What is photosynthesis?',
│   │   │   'retrieved_contexts': [
│   │   │   │   'Photosynthesis is a process used by plants to convert light energy into chemical energy.'
│   │   │   ],
│   │   │   'response': 'Photosynthesis is the process by which plants convert sunlight into energy.',
│   │   │   'reference': 'Photosynthesis is the process by which plants and other organisms convert light energy into chemical energy.'
│   │   }
│   ],
│   scores={
│   │   'answer_relevancy': ScoringResult(
│   │   │   aggregated_results={'answer_relevancy': 0.9580660217188969},
│   │   │   score_rows=[{'score': 1.0}, {'score': 0.8741980651566914}, {'score': 0.999999999999999}]
│   │   )
│   }
)

## Results Summary

In [13]:
scores = inline_results.scores["answer_relevancy"]

results_df = pd.DataFrame(
    {
        "question": [g["user_input"] for g in inline_results.generations],
        "response": [g["response"] for g in inline_results.generations],
        "score": [r["score"] for r in scores.score_rows],
    }
)

print("=" * 60)
print("RAGAS Inline Evaluation Results")
print(f"Metric: answer_relevancy")
print(f"LLM: {llm_model}")
print(f"Embedding: {embedding_model}")
print("=" * 60)
print()
print(results_df.to_string(index=False))
print()
print(f"Aggregated: {scores.aggregated_results}")

RAGAS Inline Evaluation Results
Metric: answer_relevancy
LLM: vllm-inference/Gemma-3-27B-BF16-Distributed
Embedding: vllm-embedding/qwen3-4b-embedding

                      question                                                                    response    score
What is the capital of France?                                             The capital of France is Paris. 1.000000
   Who invented the telephone?                       Alexander Graham Bell invented the telephone in 1876. 0.874198
       What is photosynthesis? Photosynthesis is the process by which plants convert sunlight into energy. 1.000000

Aggregated: {'answer_relevancy': 0.9580660217188969}
